# 06 - RAG Knowledge Retrieval

Build a multilingual FAISS retrieval index over English company policy documents. German queries can retrieve relevant English policy chunks by using multilingual sentence embeddings.

In [1]:
import os
import pandas as pd

from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

C:\Users\Shivam\AppData\Local\Temp\ipykernel_22248\1291366310.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader


In [2]:
POLICY_DIR = "../docs/company_policies"
VECTORSTORE_DIR = "../vectorstore/faiss_policy_index_multilingual"
REPORT_PATH = "../reports/rag_multilingual_test_results.csv"

os.makedirs("../reports", exist_ok=True)
os.makedirs("../vectorstore", exist_ok=True)

In [3]:
policy_files = sorted(os.listdir(POLICY_DIR))
policy_files

['account_policy.txt',
 'refund_policy.txt',
 'shipping_policy.txt',
 'technical_support_policy.txt',
 'warranty_policy.txt']

In [4]:
loader = DirectoryLoader(
    POLICY_DIR,
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
)

documents = loader.load()
print("Number of documents loaded:", len(documents))

Number of documents loaded: 5


In [5]:
for doc in documents:
    print("=" * 80)
    print("Source:", doc.metadata.get("source"))
    print(doc.page_content[:500])

Source: ..\docs\company_policies\account_policy.txt
Account Policy

Customers can request help for account login, password reset, profile update, email change, and account recovery.

For login issues:
- Verify the registered email address.
- Ask the customer to reset their password.
- Check whether the account is locked.
- Escalate if the account may be compromised.

For email change requests, support agents must verify customer identity.

For security issues, support agents should not share sensitive account information without verification.

Ac
Source: ..\docs\company_policies\refund_policy.txt
Refund Policy

Customers can request a refund within 30 days of purchase.

Refunds are allowed when:
- The product is damaged on arrival.
- The wrong item was delivered.
- The product does not match the description.
- The customer was charged incorrectly.
- The customer received a defective item.

Refunds are not allowed when:
- The product was damaged because of customer misuse.
- The refund 

In [6]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=80,
)

chunks = text_splitter.split_documents(documents)
print("Number of chunks created:", len(chunks))

Number of chunks created: 10


In [7]:
EMBEDDING_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

embedding_model = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [8]:
vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model,
)

print("Multilingual FAISS vectorstore created successfully.")

Multilingual FAISS vectorstore created successfully.


## Quick English/German Retrieval Smoke Test

In [9]:
test_queries = [
    "My product arrived damaged and I want a refund.",
    "Ich habe ein beschädigtes Produkt erhalten und möchte eine Rückerstattung.",
    "My order has not arrived and tracking is not updating.",
    "Mein Paket ist noch nicht angekommen und die Sendungsverfolgung aktualisiert sich nicht.",
    "My laptop battery drains very quickly.",
    "Der Akku meines Laptops entlädt sich sehr schnell.",
    "I cannot login to my account.",
    "Ich kann mich nicht in mein Konto einloggen.",
    "My charger stopped working within warranty period.",
    "Mein Ladegerät funktioniert innerhalb der Garantiezeit nicht mehr.",
]

for query in test_queries:
    results = vectorstore.similarity_search(query, k=1)
    print("=" * 100)
    print("Ticket:", query)
    print("Retrieved Source:", results[0].metadata.get("source"))
    print(results[0].page_content[:500])

Ticket: My product arrived damaged and I want a refund.
Retrieved Source: ..\docs\company_policies\refund_policy.txt
Refunds are not allowed when:
- The product was damaged because of customer misuse.
- The refund request is made after 30 days.
- The customer cannot provide proof of purchase.
- The item was purchased from an unauthorized seller.

For damaged products, customers may choose either a refund or replacement.

Refund requests should be assigned to the Billing or Customer Support queue depending on the issue.
High-value refund disputes should be marked as high priority.
Ticket: Ich habe ein beschädigtes Produkt erhalten und möchte eine Rückerstattung.
Retrieved Source: ..\docs\company_policies\refund_policy.txt
Refunds are not allowed when:
- The product was damaged because of customer misuse.
- The refund request is made after 30 days.
- The customer cannot provide proof of purchase.
- The item was purchased from an unauthorized seller.

For damaged products, customers may c

## Multilingual RAG Evaluation

In [10]:
multilingual_test_cases = [
    {"ticket": "My product arrived damaged and I want a refund.", "expected_policy": "refund_policy.txt", "language": "en"},
    {"ticket": "Ich habe ein beschädigtes Produkt erhalten und möchte eine Rückerstattung.", "expected_policy": "refund_policy.txt", "language": "de"},
    {"ticket": "My order has not arrived and tracking is not updating.", "expected_policy": "shipping_policy.txt", "language": "en"},
    {"ticket": "Mein Paket ist noch nicht angekommen und die Sendungsverfolgung aktualisiert sich nicht.", "expected_policy": "shipping_policy.txt", "language": "de"},
    {"ticket": "I cannot login to my account.", "expected_policy": "account_policy.txt", "language": "en"},
    {"ticket": "Ich kann mich nicht in mein Konto einloggen.", "expected_policy": "account_policy.txt", "language": "de"},
    {"ticket": "My charger stopped working within warranty period.", "expected_policy": "warranty_policy.txt", "language": "en"},
    {"ticket": "Mein Ladegerät funktioniert innerhalb der Garantiezeit nicht mehr.", "expected_policy": "warranty_policy.txt", "language": "de"},
    {"ticket": "My phone battery drains very quickly.", "expected_policy": "technical_support_policy.txt", "language": "en"},
    {"ticket": "Der Akku meines Telefons entlädt sich sehr schnell.", "expected_policy": "technical_support_policy.txt", "language": "de"},
]

rag_results = []

for case in multilingual_test_cases:
    retrieved_docs = vectorstore.similarity_search(case["ticket"], k=1)
    retrieved_source = retrieved_docs[0].metadata.get("source", "unknown_source")
    retrieved_policy = os.path.basename(retrieved_source)

    rag_results.append({
        "ticket": case["ticket"],
        "language": case["language"],
        "expected_policy": case["expected_policy"],
        "retrieved_policy": retrieved_policy,
        "is_correct": case["expected_policy"] == retrieved_policy,
        "retrieved_text": retrieved_docs[0].page_content,
    })

rag_results_df = pd.DataFrame(rag_results)
rag_results_df

,ticket,language,expected_policy,retrieved_policy,is_correct,retrieved_text
0,My product arrived damaged and I want a refund.,en,refund_policy.txt,refund_policy.txt,True,Refunds are not allowed when:\n- The product w...
1,Ich habe ein beschädigtes Produkt erhalten und...,de,refund_policy.txt,refund_policy.txt,True,Refunds are not allowed when:\n- The product w...
2,My order has not arrived and tracking is not u...,en,shipping_policy.txt,shipping_policy.txt,True,Shipping Policy\n\nStandard delivery usually t...
3,Mein Paket ist noch nicht angekommen und die S...,de,shipping_policy.txt,shipping_policy.txt,True,Shipping Policy\n\nStandard delivery usually t...
4,I cannot login to my account.,en,account_policy.txt,account_policy.txt,True,Account Policy\n\nCustomers can request help f...
5,Ich kann mich nicht in mein Konto einloggen.,de,account_policy.txt,account_policy.txt,True,Account Policy\n\nCustomers can request help f...
6,My charger stopped working within warranty per...,en,warranty_policy.txt,warranty_policy.txt,True,Warranty Policy\n\nProducts include a 1-year l...
7,Mein Ladegerät funktioniert innerhalb der Gara...,de,warranty_policy.txt,warranty_policy.txt,True,Warranty Policy\n\nProducts include a 1-year l...
8,My phone battery drains very quickly.,en,technical_support_policy.txt,warranty_policy.txt,False,Warranty Policy\n\nProducts include a 1-year l...
9,Der Akku meines Telefons entlädt sich sehr sch...,de,technical_support_policy.txt,warranty_policy.txt,False,Warranty Policy\n\nProducts include a 1-year l...


In [11]:
accuracy = rag_results_df["is_correct"].mean()
print("Multilingual RAG Retrieval Accuracy:", accuracy)

rag_results_df.to_csv(REPORT_PATH, index=False)
print("Saved report to:", REPORT_PATH)

Multilingual RAG Retrieval Accuracy: 0.8
Saved report to: ../reports/rag_multilingual_test_results.csv


## Save and Reload FAISS Index

In [12]:
vectorstore.save_local(VECTORSTORE_DIR)
print("Vectorstore saved at:", VECTORSTORE_DIR)

Vectorstore saved at: ../vectorstore/faiss_policy_index_multilingual


In [13]:
loaded_vectorstore = FAISS.load_local(
    VECTORSTORE_DIR,
    embeddings=embedding_model,
    allow_dangerous_deserialization=True,
)

print("Vectorstore loaded successfully.")

Vectorstore loaded successfully.


In [14]:
query = "Ich möchte eine Rückerstattung für ein beschädigtes Produkt."
results = loaded_vectorstore.similarity_search(query, k=2)

for i, doc in enumerate(results, start=1):
    print("=" * 80)
    print(f"Result {i}")
    print("Source:", doc.metadata.get("source"))
    print(doc.page_content[:500])

Result 1
Source: ..\docs\company_policies\refund_policy.txt
Refunds are not allowed when:
- The product was damaged because of customer misuse.
- The refund request is made after 30 days.
- The customer cannot provide proof of purchase.
- The item was purchased from an unauthorized seller.

For damaged products, customers may choose either a refund or replacement.

Refund requests should be assigned to the Billing or Customer Support queue depending on the issue.
High-value refund disputes should be marked as high priority.
Result 2
Source: ..\docs\company_policies\warranty_policy.txt
Warranty Policy

Products include a 1-year limited warranty from the date of purchase.

Warranty covers:
- Manufacturing defects.
- Hardware failure under normal use.
- Battery or charging defects.
- Product malfunction not caused by misuse.
- Device failure within the warranty period.

Warranty does not cover:
- Physical damage caused by the customer.
- Water damage.
- Unauthorized repair.
- Accidental d